In [1]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
import os

# Load environment variables from .env file
load_dotenv()

llm_openai = ChatOpenAI(model = "deepseek-chat",
base_url="https://api.deepseek.com",
api_key=os.environ.get('DEEPSEEK_API_KEY'),
 temperature=0)


if os.environ.get('DEEPSEEK_API_KEY') :
    print("DEEPSEEK_API_KEY is set")
else:    print("DEEPSEEK_API_KEY is not set")




KeyboardInterrupt: 

In [ ]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_Flag: Literal["正面","反面"] 



# **CHAIN WITH Conditional Chains**

In [ ]:
# TASK-1 prompt

prompt_template = ChatPromptTemplate.from_messages(
[("system", "你是一个电影评价的评估者"),
("human", "请将这个电影的评价区分为正面或反面：{input}")])

In [ ]:
# TASK-2  LLM


# llm_openai = ChatOpenAI(model = "deepseek-chat",
# base_url="https://api.deepseek.com",
# api_key=os.environ.get('DEEPSEEK_API_KEY'),
#  temperature=0)

llm_structured_output = llm_openai.with_structured_output(llm_schema,
method = "function_calling"
)


In [ ]:
# TASK-3  StrParser

# str_parser = StrOutputParser()

In [ ]:
# TASK-4  [Custom Runnable]  
from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) ->str:
  return input.model_dump()['movie_summary_Flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)  

### **conditional CHAIN 1**

In [ ]:
# TASK - 1 [prompt]


linked_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的微博电影推荐博主，擅长写热情、燃点高、带感染力的正面电影帖子。风格简短有力，可加 emoji 和 hashtag。"),
    ("human", "电影评价结果是「正面」。请根据以下电影内容，生成一条热情洋溢的微博帖子：\n{text}")
])

# TASK - 2 [LLM] llm_openai = ChatOpenAI(model = "deepseek-chat",

# TASK - 3 [StrParser] 
str_parser = StrOutputParser()

chain_linked = linked_prompt | llm_openai | str_parser

### **conditional CHAIN 2**

In [ ]:
from langchain_core.runnables import RunnableParallel , RunnableBranch

In [ ]:
def insta_chain(text: dict) :

 text = text["text"]

 # TASK - 1 [prompt]
 insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的小红书电影分享博主，文风真诚、自然、治愈或燃爆，适合女生/电影爱好者，多用 emoji、分段和个人感受。"),
    ("human", "电影评价结果是「正面」。请根据以下电影内容，生成一条小红书风格的电影推荐笔记：\n{text}")
])
 # TASK - 2 [LLM] llm_openai = ChatOpenAmodel =     "deepseek-chat",
 # TASK - 3 [StrParser] str_parser =trOutputParser()

 chain_insta = insta_prompt | llm_openai | str_parser

 result =  chain_insta.invoke(text)

 return result

insta_chain_runnable = RunnableLambda(insta_chain)

### **Final Orchestration**

In [ ]:
def normalize_input(x):
    if isinstance(x, str):
        return {"text": x}
    elif isinstance(x, dict):
        # 兼容 "input" 和 "text"
        if "text" not in x and "input" in x:
            x = x.copy()
            x["text"] = x["input"]
        return x
    return {"text": str(x)}

# ====================== Conditional Chain ======================
conditional_chain = RunnableBranch(
    # 条件1：如果是“正面”评价 → 生成微博帖子
    (lambda x: "正面" in str(x), chain_linked),
    
    # 默认分支（其他情况，包括“反面”）→ 生成小红书帖子
    # 注意：这里必须是单个 Runnable，不能直接写 insta_chain_runnable（要处理输入）
    insta_chain_runnable
)

# ====================== Final Orchestrator ======================
final_orschestrator = (
    prompt_template 
    | llm_structured_output 
    | pydantic_json_runnable 
    | RunnableLambda(normalize_input)   # ← 关键：统一输入格式
    | conditional_chain
)

In [ ]:
final_orschestrator.invoke({"input": "i do not love kgf movie"})

'🎬 这部电影真的让我哭到不行！😭 后劲太大了！\n\n✨ 剧情完全超出预期\n本以为是个普通的故事\n没想到每个转折都直击心灵\n导演真的太会拍了！\n\n🌟 演员演技全员在线\n特别是女主角的哭戏\n每一滴眼泪都流进了我心里\n看完后久久不能平复\n\n💫 最打动我的是那句台词：\n“爱不是占有，而是成全”\n瞬间破防了谁懂啊！\n\n🌈 虽然有些情节很虐心\n但整体传递的温暖和希望\n真的治愈了我最近的低落情绪\n强烈推荐给需要被治愈的你！\n\n👭 适合和闺蜜一起看\n记得备好纸巾！\n看完我们聊到凌晨三点\n每个人都有不同的感悟\n\n#治愈系电影 #女生必看 #后劲太大 #电影推荐\n\n（悄悄说：片尾有彩蛋哦！别急着关！）🎉'